# ⚡ Adobe Stock AI Studio
### Google Colab Launcher — One notebook, complete system

**What this notebook does (run ALL cells once):**

1. Detects T4 GPU
2. Mounts Google Drive
3. Installs system dependencies (including zstd for Ollama)
4. Installs Ollama (local vision AI — no external API)
5. Pulls and tests a vision model
6. Installs Python dependencies
7. Writes all application files to disk
8. Downloads Real-ESRGAN weights
9. Starts FastAPI backend
10. Opens Cloudflare tunnel

**Then opens a single web app URL where you:**
- Drop all your images (upload-first)
- Click Start Processing
- Every image is upscaled → Ollama analyzes it → metadata saved
- Download: `AdobeStock_Metadata.csv`, `AdobeStock_Metadata.json`, ZIP

### Quick start
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all** (`Ctrl+F9`)
3. Wait ~10 min for first-time setup
4. Click the URL shown in the last cell

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 2: T4 GPU Detection
# ═══════════════════════════════════════════════════════
import torch, sys

print("=" * 55)
print("  ADOBE STOCK AI STUDIO — GPU Check")
print("=" * 55)

if not torch.cuda.is_available():
    print("❌  No GPU detected!")
    print("   Go to Runtime → Change runtime type → T4 GPU → Save")
    raise SystemExit("GPU required.")

gpu_name = torch.cuda.get_device_name(0)
free_b, total_b = torch.cuda.mem_get_info(0)
print(f"  GPU  : {gpu_name}")
print(f"  VRAM : {free_b/1024**3:.1f} GB free / {total_b/1024**3:.1f} GB total")
print("  ✓ T4 GPU READY")
print("=" * 55)


In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 3: Google Drive — Persistence
# ═══════════════════════════════════════════════════════
from google.colab import drive
import os

mount_drive = True  #@param {type:"boolean"}

STUDIO_DIR = '/content/studio'
for sub in ['uploads','output','metadata','logs','archives','failed','temp_output']:
    os.makedirs(f'{STUDIO_DIR}/{sub}', exist_ok=True)

if mount_drive:
    print("Mounting Google Drive…")
    drive.mount('/content/drive')
    drive_path = '/content/drive/MyDrive/AdobeStockStudio'
    for sub in ['uploads','output','metadata','logs','archives','failed']:
        os.makedirs(f'{drive_path}/{sub}', exist_ok=True)
    print(f"✓ Drive mounted → {drive_path}")
else:
    print("Drive bypass — using local Colab storage only.")

print("✓ Directory structure ready")


In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 4: System Dependencies (zstd must be FIRST)
# ═══════════════════════════════════════════════════════
import subprocess, sys

def apt(pkgs):
    subprocess.run(['apt-get','install','-y','-qq'] + pkgs,
                   capture_output=True)

print("Installing system dependencies…")
subprocess.run(['apt-get','update','-qq'], capture_output=True)

# zstd MUST be installed before Ollama
apt(['zstd'])
print("  ✓ zstd")

apt(['curl','wget','pv','libgl1-mesa-glx','libglib2.0-0'])
print("  ✓ curl, wget, libgl1, libglib2")
print("✓ System dependencies ready")


In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 5: Install Ollama (local AI — no external API)
# ═══════════════════════════════════════════════════════
import subprocess, shutil

if not shutil.which('ollama'):
    print("Installing Ollama…")
    result = subprocess.run(
        'curl -fsSL https://ollama.com/install.sh | sh',
        shell=True, capture_output=True, text=True
    )
    if result.returncode != 0:
        print("STDERR:", result.stderr[-500:])
        raise RuntimeError("Ollama installation failed")
    print("✓ Ollama installed")
else:
    print(f"✓ Ollama already installed: {shutil.which('ollama')}")


In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 6: Start Ollama Server
# ═══════════════════════════════════════════════════════
import subprocess, time, requests

OLLAMA_HOST = 'http://127.0.0.1:11434'

# Check if already running
try:
    if requests.get(f'{OLLAMA_HOST}/api/tags', timeout=2).status_code == 200:
        print("✓ Ollama already running")
        ollama_proc = None
    else:
        raise Exception()
except Exception:
    print("Starting Ollama server…")
    ollama_proc = subprocess.Popen(
        ['ollama', 'serve'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )

    # Wait for readiness
    for i in range(40):
        try:
            if requests.get(f'{OLLAMA_HOST}/api/tags', timeout=2).ok:
                print(f"✓ Ollama server ready (took {i+1}s)")
                break
        except Exception:
            pass
        time.sleep(1)
    else:
        raise RuntimeError("Ollama server did not start within 40s")


In [ ]:
# =======================================================
# CELL 7: Vision Model — Detect -> Pull -> Validate -> Fallback
# =======================================================
import requests, json, base64, io, time, os
from PIL import Image, ImageDraw

OLLAMA_HOST = os.environ.get('OLLAMA_HOST', 'http://127.0.0.1:11434')
OLLAMA_VISION_MODELS = [
    'moondream',
    'moondream:latest',
    'llava:7b',
    'llava',
    'llava:latest',
    'bakllava',
    'llama3.2-vision',
    'minicpm-v',
]

print("=" * 55)
print("  OLLAMA VISION MODEL VALIDATION")
print("=" * 55)

# ── 1. Create Patterned Test Image (256x256) ─────────────────────
# Guarantees vision token activation across SigLIP / CLIP encoders
test_img = Image.new('RGB', (256, 256), color=(30, 60, 120))
draw = ImageDraw.Draw(test_img)
draw.rectangle([20, 20, 100, 100], fill=(220, 80, 40), outline=(255, 255, 255))
draw.ellipse([120, 50, 220, 150], fill=(40, 180, 90), outline=(255, 255, 255))
draw.polygon([(128, 160), (60, 230), (196, 230)], fill=(240, 200, 30))
draw.line([(0, 0), (256, 256)], fill=(255, 255, 255), width=3)
buf = io.BytesIO()
test_img.save(buf, format='JPEG', quality=90)
img_b64 = base64.b64encode(buf.getvalue()).decode('ascii')

# ── 2. List Installed Models ─────────────────────────────────────
try:
    r = requests.get(f'{OLLAMA_HOST}/api/tags', timeout=10)
    installed = [m['name'] for m in r.json().get('models', []) if 'name' in m]
    print(f"Installed Ollama models: {installed or 'none'}")
except Exception as e:
    installed = []
    print(f"[WARN] Could not list models: {e}")

# ── 3. Helper: Query Vision (Chat + Generate Dual Endpoint) ──────
def query_vision_test(model_name, b64_data):
    test_prompt = "Describe the colors, shapes, and objects in this image in one or two clear sentences."

    # Try /api/chat first (Primary & standard for multimodal)
    try:
        chat_payload = {
            "model": model_name,
            "messages": [{"role": "user", "content": test_prompt, "images": [b64_data]}],
            "stream": False,
            "options": {"temperature": 0.2, "num_predict": 200}
        }
        cr = requests.post(f'{OLLAMA_HOST}/api/chat', json=chat_payload, timeout=120)
        if cr.status_code == 200:
            content = cr.json().get("message", {}).get("content", "").strip()
            if content:
                return True, content, "/api/chat"
            else:
                print(f"  [/api/chat] Returned empty content. done={cr.json().get('done')}, reason={cr.json().get('done_reason')}")
    except Exception as ce:
        print(f"  [/api/chat] Error: {ce}")

    # Try /api/generate fallback
    try:
        gen_payload = {
            "model": model_name,
            "prompt": test_prompt,
            "images": [b64_data],
            "stream": False,
            "options": {"temperature": 0.2, "num_predict": 200}
        }
        gr = requests.post(f'{OLLAMA_HOST}/api/generate', json=gen_payload, timeout=120)
        if gr.status_code == 200:
            resp_text = gr.json().get("response", "").strip()
            if resp_text:
                return True, resp_text, "/api/generate"
            else:
                print(f"  [/api/generate] Returned empty response. done={gr.json().get('done')}, reason={gr.json().get('done_reason')}")
    except Exception as ge:
        print(f"  [/api/generate] Error: {ge}")

    return False, "", "failed"

# ── 4. Candidate Validation Loop with Fallback ───────────────────
active_model = None
for candidate in OLLAMA_VISION_MODELS:
    print(f"\nTesting candidate vision model: '{candidate}'...")
    base = candidate.split(":")[0].lower()
    is_installed = any(base in inst.lower() for inst in installed)

    if not is_installed:
        print(f"Model '{candidate}' not installed. Pulling...")
        try:
            with requests.post(f'{OLLAMA_HOST}/api/pull', json={'name': candidate}, stream=True, timeout=900) as resp:
                last_pct = -1
                for line in resp.iter_lines():
                    if line:
                        try:
                            obj = json.loads(line)
                            if obj.get('total', 0) > 0:
                                pct = int(obj['completed'] / obj['total'] * 100)
                                if pct // 10 != last_pct // 10:
                                    print(f"  Pulling {candidate}: {pct}%")
                                    last_pct = pct
                        except Exception:
                            pass
            r2 = requests.get(f'{OLLAMA_HOST}/api/tags', timeout=5)
            installed = [m['name'] for m in r2.json().get('models', [])]
        except Exception as pe:
            print(f"  [WARN] Failed to pull '{candidate}': {pe}")
            continue

    # Resolve full installed tag name
    resolved_tag = candidate
    for inst in installed:
        if base in inst.lower():
            resolved_tag = inst
            break

    # Execute vision inference test
    print(f"Running image vision test on '{resolved_tag}'...")
    success, text_out, endpoint = query_vision_test(resolved_tag, img_b64)
    if success and text_out:
        snippet = text_out.replace('\n', ' ')[:90]
        print(f"[OK] Vision inference test PASSED via {endpoint} on '{resolved_tag}'")
        print(f"     Output: \"{snippet}...\"")
        active_model = resolved_tag
        break
    else:
        print(f"[WARN] Vision test failed on '{resolved_tag}'. Trying fallback model...")

# ── 5. Save Runtime Config or Final Status ───────────────────────
if active_model:
    os.makedirs('/content/studio', exist_ok=True)
    with open('/content/studio/.runtime_config.json', 'w') as f:
        json.dump({'ollama_ready': True, 'ollama_model': active_model, 'vision_tested': True}, f)
    print(f"\n[OK] Ollama Vision READY -- Active Model: {active_model}")
else:
    print("\n[ERROR] No vision model passed the image inference test.")
    print("         Please check that Ollama is running and has GPU access.")
    raise RuntimeError("Ollama vision validation failed on all candidate models.")


In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 8: Python Dependencies
# ═══════════════════════════════════════════════════════
import subprocess, sys

def pip(pkgs, quiet=True):
    q = ['-q'] if quiet else []
    subprocess.run([sys.executable, '-m', 'pip', 'install'] + q + pkgs, check=True)

print("Installing Python packages…")

pip(['fastapi>=0.95', 'uvicorn[standard]>=0.22', 'python-multipart>=0.0.6'])
print("  ✓ fastapi, uvicorn")

pip(['httpx>=0.24', 'aiofiles>=23', 'pillow>=10', 'psutil>=5.9', 'pydantic>=2'])
print("  ✓ httpx, pillow, psutil, pydantic")

pip(['basicsr', 'facexlib', 'gfpgan', 'realesrgan>=0.3'])
print("  ✓ basicsr, realesrgan")

pip(['opencv-python-headless'])
print("  ✓ opencv-python-headless")

# torchvision compat patch
import sys as _sys
try:
    import torchvision.transforms.functional as _F
    _sys.modules['torchvision.transforms.functional_tensor'] = _F
    print("  ✓ torchvision compat patch")
except Exception: pass

print("✓ Python dependencies installed")


In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 9: Patch basicsr (functional_tensor compat)
# ═══════════════════════════════════════════════════════
import os, sys

# Apply sys.modules patch first
try:
    import torchvision.transforms.functional as _F
    sys.modules['torchvision.transforms.functional_tensor'] = _F
except Exception: pass

# Patch degradations.py if needed
try:
    import basicsr
    deg_path = os.path.join(
        os.path.dirname(basicsr.__file__), 'data', 'degradations.py'
    )
    if os.path.exists(deg_path):
        content = open(deg_path).read()
        if 'functional_tensor' in content:
            patched = content.replace(
                'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
                'from torchvision.transforms.functional import rgb_to_grayscale'
            )
            open(deg_path, 'w').write(patched)
            print("✓ basicsr degradations.py patched")
        else:
            print("✓ basicsr does not need patching")
except Exception as e:
    print(f"  Note: basicsr patch: {e}")

print("✓ basicsr ready")


In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 10: Real-ESRGAN Weights + Inference Script
# ═══════════════════════════════════════════════════════
import os, subprocess, shutil

STUDIO = '/content/studio'
WEIGHTS_DIR = f'{STUDIO}/experiments/pretrained_models'
os.makedirs(WEIGHTS_DIR, exist_ok=True)

# Model weights
WEIGHT_FILE = f'{WEIGHTS_DIR}/RealESRGAN_x4plus.pth'
if not os.path.exists(WEIGHT_FILE):
    print("Downloading RealESRGAN_x4plus weights (~67 MB)…")
    subprocess.run([
        'wget', '-q', '-O', WEIGHT_FILE,
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'
    ], check=True)
    print("✓ Weights downloaded")
else:
    print(f"✓ Weights present: {WEIGHT_FILE}")

# Clone Real-ESRGAN for inference script
RESRGAN_DIR = '/content/Real-ESRGAN'
if not os.path.exists(RESRGAN_DIR):
    print("Cloning Real-ESRGAN repo…")
    subprocess.run(
        ['git', 'clone', '--depth=1',
         'https://github.com/xinntao/Real-ESRGAN.git', RESRGAN_DIR],
        check=True, capture_output=True
    )
    print("✓ Real-ESRGAN cloned")
else:
    print("[OK] Real-ESRGAN cloned")
else:
    print("[OK] Real-ESRGAN already present")

# Link inference script
INFER_DST = f'{STUDIO}/inference_realesrgan.py'
INFER_SRC = f'{RESRGAN_DIR}/inference_realesrgan.py'
if not os.path.exists(INFER_DST) and os.path.exists(INFER_SRC):
    shutil.copy(INFER_SRC, INFER_DST)
    print("[OK] inference_realesrgan.py linked")

print("[OK] Real-ESRGAN ready")


In [ ]:
# =======================================================
# CELL 11: Write All Application Scripts to Disk
# =======================================================
import base64, os
STUDIO = '/content/studio'
os.makedirs(f'{STUDIO}/scripts', exist_ok=True)
os.makedirs(f'{STUDIO}/app', exist_ok=True)

# scripts/__init__.py
_b64 = 'IyBBZG9iZSBTdG9jayBBSSBTdHVkaW8K'
os.makedirs('/content/studio/scripts', exist_ok=True)
open('/content/studio/scripts/__init__.py', 'w', encoding='utf-8').write(base64.b64decode(_b64).decode())
print('  [OK] scripts/__init__.py')

# scripts/config.py
_b64 = 'aW1wb3J0IG9zCgojIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAojIEFkb2JlIFN0b2NrIEFJIFN0dWRpbyDigJQgQ2VudHJhbGl6ZWQgQ29uZmlndXJhdGlvbgojIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKIyBBcHAgaWRlbnRpdHkKQVBQX05BTUUgPSAiQWRvYmUgU3RvY2sgQUkgU3R1ZGlvIgpBUFBfVkVSU0lPTiA9ICIyLjAuMCIKCiMgRGVmYXVsdCB1cHNjYWxpbmcgc2V0dGluZ3MKREVGQVVMVF9TQ0FMRSA9IDQKREVGQVVMVF9UQVJHRVRfV0lEVEggPSAzODQwCkRFRkFVTFRfVEFSR0VUX0hFSUdIVCA9IDIxNjAKSlBFR19RVUFMSVRZID0gOTUKREVGQVVMVF9GT1JNQVQgPSAianBnIgpNT0RFTF9OQU1FID0gIlJlYWxFU1JHQU5feDRwbHVzIgpNQVhfVVBMT0FEX1NJWkVfTUIgPSAxMDAKTUFYX0NPTkNVUlJFTlRfVVBMT0FEUyA9IDQKCiMgVGlsaW5nICYgVlJBTSBvcHRpbWl6YXRpb24gcGFyYW1ldGVycyAocHJvdGVjdHMgVDQgZnJvbSBPT00pClRJTEVfU0laRSA9IDQwMApUSUxFX1BBRCA9IDEwClBSRV9QQUQgPSAxMAoKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKIyBPbGxhbWEgQ29uZmlndXJhdGlvbgojIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgApPTExBTUFfSE9TVCA9IG9zLmVudmlyb24uZ2V0KCJPTExBTUFfSE9TVCIsICJodHRwOi8vMTI3LjAuMC4xOjExNDM0IikKT0xMQU1BX1ZJU0lPTl9NT0RFTCA9IG9zLmVudmlyb24uZ2V0KCJPTExBTUFfVklTSU9OX01PREVMIiwgIm1vb25kcmVhbSIpCk9MTEFNQV9USU1FT1VUID0gaW50KG9zLmVudmlyb24uZ2V0KCJPTExBTUFfVElNRU9VVCIsIDE4MCkpICAjIDE4MCBzZWNvbmRzIHBlciB2aXNpb24gaW5mZXJlbmNlIGNhbGwKT0xMQU1BX01BWF9SRVRSSUVTID0gMwpPTExBTUFfSlNPTl9NQVhfUkVUUklFUyA9IDMKCiMgUHJlZmVycmVkIHZpc2lvbiBtb2RlbHMgaW4gZmFsbGJhY2sgb3JkZXIgKHRlc3RlZCB3aXRoIGFjdHVhbCBpbWFnZSBpbnB1dHMpCk9MTEFNQV9WSVNJT05fTU9ERUxTID0gWwogICAgIm1vb25kcmVhbSIsCiAgICAibW9vbmRyZWFtOmxhdGVzdCIsCiAgICAibGxhdmE6N2IiLAogICAgImxsYXZhIiwKICAgICJsbGF2YTpsYXRlc3QiLAogICAgImJha2xsYXZhIiwKICAgICJsbGFtYTMuMi12aXNpb24iLAogICAgIm1pbmljcG0tdiIsCl0KT0xMQU1BX01PREVMX1BVTExfVElNRU9VVCA9IDkwMCAgIyAxNSBtaW4gbWF4IHB1bGwgdGltZQoKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKIyBNZXRhZGF0YSAvIENTViBvdXRwdXQgc2V0dGluZ3MKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKTUFYX0tFWVdPUkRTID0gNDkKTUFYX1RJVExFX0xFTkdUSCA9IDIwMApDU1ZfRklMRU5BTUUgPSAiQWRvYmVTdG9ja19NZXRhZGF0YS5jc3YiCkpTT05fRklMRU5BTUUgPSAiQWRvYmVTdG9ja19NZXRhZGF0YS5qc29uIgpDU1ZfQ09MVU1OUyA9IFsiRmlsZW5hbWUiLCAiVGl0bGUiLCAiS2V5d29yZHMiLCAiQ2F0ZWdvcnkiLCAiUmVsZWFzZXMiXQoKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKIyBBZG9iZSBTdG9jayBudW1lcmljIGNhdGVnb3J5IG1hcAojIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgApBRE9CRV9DQVRFR09SWV9NQVAgPSB7CiAgICAiYW5pbWFscyI6IDEsCiAgICAiYnVpbGRpbmdzIjogMiwKICAgICJidXNpbmVzcyI6IDMsCiAgICAiZHJpbmtzIjogNCwKICAgICJlbnZpcm9ubWVudCI6IDUsCiAgICAic3RhdGVzX29mX21pbmQiOiA2LAogICAgImZvb2QiOiA3LAogICAgImdyYXBoaWNfcmVzb3VyY2VzIjogOCwKICAgICJob2JiaWVzX2FuZF9sZWlzdXJlIjogOSwKICAgICJpbmR1c3RyeSI6IDEwLAogICAgImxhbmRzY2FwZSI6IDExLAogICAgImxpZmVzdHlsZSI6IDEyLAogICAgInBlb3BsZSI6IDEzLAogICAgInBsYW50c19hbmRfZmxvd2VycyI6IDE0LAogICAgImN1bHR1cmVfYW5kX3JlbGlnaW9uIjogMTUsCiAgICAic2NpZW5jZSI6IDE2LAogICAgInNvY2lhbF9pc3N1ZXMiOiAxNywKICAgICJzcG9ydHMiOiAxOCwKICAgICJ0ZWNobm9sb2d5IjogMTksCiAgICAidHJhbnNwb3J0IjogMjAsCiAgICAidHJhdmVsIjogMjEsCiAgICAiYWJzdHJhY3QiOiAyMiwKfQoKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKIyBEaXJlY3RvcnkgcGF0aHMKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKRFJJVkVfTU9VTlRfUEFSRU5UID0gIi9jb250ZW50L2RyaXZlL015RHJpdmUiCkRSSVZFX1BST0pFQ1RfUEFUSCA9IG9zLnBhdGguam9pbihEUklWRV9NT1VOVF9QQVJFTlQsICJBZG9iZVN0b2NrU3R1ZGlvIikKCkxPQ0FMX0JBU0UgPSAiLi9BZG9iZVN0b2NrU3R1ZGlvIgoKVEVNUF9JTlBVVF9ESVIgPSBvcy5wYXRoLmpvaW4oTE9DQUxfQkFTRSwgInVwbG9hZHMiKQpURU1QX09VVFBVVF9ESVIgPSBvcy5wYXRoLmpvaW4oTE9DQUxfQkFTRSwgInRlbXBfb3V0cHV0IikKCgpkZWYgcmVzb2x2ZV9wYXRocygpOgogICAgIiIiCiAgICBEeW5hbWljYWxseSBtYXBzIGZvbGRlcnMgdG8gR29vZ2xlIERyaXZlIGlmIG1vdW50ZWQsCiAgICBmYWxsaW5nIGJhY2sgdG8gbG9jYWwgc3RvcmFnZSBpZiBEcml2ZSBpcyBub3QgbW91bnRlZC4KICAgICIiIgogICAgaWYgb3MucGF0aC5leGlzdHMoRFJJVkVfTU9VTlRfUEFSRU5UKToKICAgICAgICBiYXNlID0gRFJJVkVfUFJPSkVDVF9QQVRICiAgICBlbHNlOgogICAgICAgIGJhc2UgPSBMT0NBTF9CQVNFCgogICAgcmV0dXJuIHsKICAgICAgICAidXBsb2FkcyI6IG9zLnBhdGguam9pbihiYXNlLCAidXBsb2FkcyIpLAogICAgICAgICJvdXRwdXQiOiBvcy5wYXRoLmpvaW4oYmFzZSwgIm91dHB1dCIpLAogICAgICAgICJtZXRhZGF0YSI6IG9zLnBhdGguam9pbihiYXNlLCAibWV0YWRhdGEiKSwKICAgICAgICAiZmFpbGVkIjogb3MucGF0aC5qb2luKGJhc2UsICJmYWlsZWQiKSwKICAgICAgICAibG9ncyI6IG9zLnBhdGguam9pbihiYXNlLCAibG9ncyIpLAogICAgICAgICJhcmNoaXZlcyI6IG9zLnBhdGguam9pbihiYXNlLCAiYXJjaGl2ZXMiKSwKICAgIH0KCgpkZWYgZW5zdXJlX2RpcnMoKToKICAgICIiIkNyZWF0ZSBhbGwgcmVxdWlyZWQgZGlyZWN0b3JpZXMuIiIiCiAgICBwYXRocyA9IHJlc29sdmVfcGF0aHMoKQogICAgZm9yIHAgaW4gcGF0aHMudmFsdWVzKCk6CiAgICAgICAgb3MubWFrZWRpcnMocCwgZXhpc3Rfb2s9VHJ1ZSkKICAgIG9zLm1ha2VkaXJzKFRFTVBfSU5QVVRfRElSLCBleGlzdF9vaz1UcnVlKQogICAgb3MubWFrZWRpcnMoVEVNUF9PVVRQVVRfRElSLCBleGlzdF9vaz1UcnVlKQogICAgcmV0dXJuIHBhdGhzCg=='
os.makedirs('/content/studio/scripts', exist_ok=True)
open('/content/studio/scripts/config.py', 'w', encoding='utf-8').write(base64.b64decode(_b64).decode())
print('  [OK] scripts/config.py')

# scripts/utils.py
_b64 = 'IiIiCnNjcmlwdHMvdXRpbHMucHkg4oCUIEFkb2JlIFN0b2NrIEFJIFN0dWRpbwoKU3lzdGVtIHJlc291cmNlIHV0aWxpdGllcy4KUHJlc2VydmVkIGZyb20gb3JpZ2luYWwuCiIiIgoKaW1wb3J0IG9zCmltcG9ydCBwc3V0aWwKCnRyeToKICAgIGltcG9ydCB0b3JjaApleGNlcHQgSW1wb3J0RXJyb3I6CiAgICB0b3JjaCA9IE5vbmUKCgpkZWYgZ2V0X3N5c3RlbV9yZXNvdXJjZXMoKSAtPiBkaWN0OgogICAgIiIiCiAgICBSZXR1cm5zIHN5c3RlbSBSQU0gYW5kIEdQVSBWUkFNIG1ldHJpY3MuCiAgICAiIiIKICAgIGdwdV9hdmFpbGFibGUgPSBGYWxzZQogICAgZ3B1X25hbWUgPSAiTm9uZSIKICAgIHZyYW1faW5mbyA9IHsiZnJlZSI6IDAuMCwgInRvdGFsIjogMC4wLCAidXNlZCI6IDAuMH0KCiAgICBpZiB0b3JjaCBhbmQgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICBncHVfYXZhaWxhYmxlID0gVHJ1ZQogICAgICAgIHRyeToKICAgICAgICAgICAgZ3B1X25hbWUgPSB0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgwKQogICAgICAgICAgICBmcmVlX2IsIHRvdGFsX2IgPSB0b3JjaC5jdWRhLm1lbV9nZXRfaW5mbygwKQogICAgICAgICAgICB2cmFtX2luZm9bImZyZWUiXSA9IHJvdW5kKGZyZWVfYiAvICgxMDI0ICoqIDMpLCAyKQogICAgICAgICAgICB2cmFtX2luZm9bInRvdGFsIl0gPSByb3VuZCh0b3RhbF9iIC8gKDEwMjQgKiogMyksIDIpCiAgICAgICAgICAgIHZyYW1faW5mb1sidXNlZCJdID0gcm91bmQoKHRvdGFsX2IgLSBmcmVlX2IpIC8gKDEwMjQgKiogMyksIDIpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgcGFzcwoKICAgIHJhbSA9IHBzdXRpbC52aXJ0dWFsX21lbW9yeSgpCiAgICByYW1faW5mbyA9IHsKICAgICAgICAidXNlZCI6IHJvdW5kKHJhbS51c2VkIC8gKDEwMjQgKiogMyksIDIpLAogICAgICAgICJ0b3RhbCI6IHJvdW5kKHJhbS50b3RhbCAvICgxMDI0ICoqIDMpLCAyKSwKICAgICAgICAicGVyY2VudCI6IHJhbS5wZXJjZW50LAogICAgfQoKICAgIHJldHVybiB7CiAgICAgICAgImdwdSI6IGdwdV9hdmFpbGFibGUsCiAgICAgICAgImdwdV9uYW1lIjogZ3B1X25hbWUsCiAgICAgICAgInJhbV91c2FnZSI6IHJhbV9pbmZvLAogICAgICAgICJ2cmFtX3VzYWdlIjogdnJhbV9pbmZvLAogICAgfQoKCmRlZiBnZXRfdW5pcXVlX291dHB1dF9maWxlbmFtZShkaXJlY3Rvcnk6IHN0ciwgaW5kZXg6IGludCwgZXh0OiBzdHIpIC0+IHN0cjoKICAgICIiIgogICAgR2VuZXJhdGUgc3RhbmRhcmRpemVkIEFkb2JlIFN0b2NrIG91dHB1dCBmaWxlbmFtZS4KICAgIEZvcm1hdDogc3RvY2tfaW1hZ2VfdXB7Tn0ue2V4dH0KICAgIENvbGxpc2lvbi1zYWZlOiBpbmNyZW1lbnRzIGluZGV4IGlmIGZpbGUgYWxyZWFkeSBleGlzdHMuCiAgICAiIiIKICAgIHdoaWxlIFRydWU6CiAgICAgICAgbmFtZSA9IGYic3RvY2tfaW1hZ2VfdXB7aW5kZXh9LntleHR9IgogICAgICAgIHBhdGggPSBvcy5wYXRoLmpvaW4oZGlyZWN0b3J5LCBuYW1lKQogICAgICAgIGlmIG5vdCBvcy5wYXRoLmV4aXN0cyhwYXRoKToKICAgICAgICAgICAgcmV0dXJuIG5hbWUsIGluZGV4CiAgICAgICAgaW5kZXggKz0gMQoKCmRlZiBmb3JtYXRfZXRhKHNlY29uZHM6IGZsb2F0IHwgTm9uZSkgLT4gc3RyOgogICAgaWYgc2Vjb25kcyBpcyBOb25lOgogICAgICAgIHJldHVybiAiLS0iCiAgICBzZWNvbmRzID0gaW50KHNlY29uZHMpCiAgICBpZiBzZWNvbmRzIDwgNjA6CiAgICAgICAgcmV0dXJuIGYie3NlY29uZHN9cyIKICAgIG1pbnV0ZXMgPSBzZWNvbmRzIC8vIDYwCiAgICBzZWNzID0gc2Vjb25kcyAlIDYwCiAgICByZXR1cm4gZiJ7bWludXRlc31tIHtzZWNzfXMiCg=='
os.makedirs('/content/studio/scripts', exist_ok=True)
open('/content/studio/scripts/utils.py', 'w', encoding='utf-8').write(base64.b64decode(_b64).decode())
print('  [OK] scripts/utils.py')

# scripts/qc.py
_b64 = 'IiIiCnNjcmlwdHMvcWMucHkg4oCUIEFkb2JlIFN0b2NrIEFJIFN0dWRpbwoKVGVjaG5pY2FsIFF1YWxpdHkgQ29udHJvbCBmb3IgdXBzY2FsZWQgaW1hZ2VzLgpQcmVzZXJ2ZWQgZnJvbSBvcmlnaW5hbCB3aXRoIDYgdmVyaWZpY2F0aW9uIGNoZWNrcy4KIiIiCgppbXBvcnQgb3MKaW1wb3J0IGxvZ2dpbmcKZnJvbSBQSUwgaW1wb3J0IEltYWdlCgpsb2dnZXIgPSBsb2dnaW5nLmdldExvZ2dlcigiQWRvYmVTdG9ja1N0dWRpby5RQyIpCgpNSU5fTUVHQVBJWEVMUyA9IDQuMCAgICAgICAgICAjIEFkb2JlIFN0b2NrIG1pbmltdW0KTUFYX0ZJTEVfU0laRV9NQiA9IDQ1LjAgICAgICAgICMgQWRvYmUgU3RvY2sgdXBsb2FkIGxpbWl0Ck1BWF9BU1BFQ1RfUkFUSU9fRFJJRlQgPSAwLjAyICAjIDIlIHRvbGVyYW5jZQoKCmRlZiBydW5fdGVjaG5pY2FsX3FjKAogICAgb3V0cHV0X3BhdGg6IHN0ciwKICAgIG9yaWdpbmFsX3c6IGludCwKICAgIG9yaWdpbmFsX2g6IGludCwKICAgIHJlcV9mb3JtYXQ6IHN0ciwKKSAtPiBkaWN0OgogICAgIiIiCiAgICBSdW5zIDYgdGVjaG5pY2FsIFFDIGNoZWNrcyBvbiB0aGUgdXBzY2FsZWQgb3V0cHV0IGZpbGUuCgogICAgUmV0dXJuczoKICAgIHsKICAgICAgInBhc3NlZCI6IGJvb2wsCiAgICAgICJoYXJkX2ZhaWx1cmVzIjogbGlzdFtzdHJdLAogICAgICAid2FybmluZ3MiOiBsaXN0W3N0cl0sCiAgICAgICJpbnB1dCI6IHsid2lkdGgiOiBpbnQsICJoZWlnaHQiOiBpbnR9LAogICAgICAib3V0cHV0IjogeyJ3aWR0aCI6IGludCwgImhlaWdodCI6IGludH0sCiAgICAgICJtZWdhcGl4ZWxzIjogZmxvYXQsCiAgICAgICJjaGVja3MiOiB7CiAgICAgICAgICAicmVzb2x1dGlvbiI6ICJwYXNzInwiZmFpbCJ8Indhcm4iLAogICAgICAgICAgImZvcm1hdCI6ICJwYXNzInwiZmFpbCIsCiAgICAgICAgICAiaW50ZWdyaXR5IjogInBhc3MifCJmYWlsIiwKICAgICAgICAgICJhc3BlY3RfcmF0aW8iOiAicGFzcyJ8Indhcm4iLAogICAgICAgICAgInRyYW5zcGFyZW5jeSI6ICJwYXNzInwid2FybiIsCiAgICAgICAgICAic2l6ZSI6ICJwYXNzInwid2FybiJ8ImZhaWwiLAogICAgICB9CiAgICB9CiAgICAiIiIKICAgIHJlc3VsdCA9IHsKICAgICAgICAicGFzc2VkIjogVHJ1ZSwKICAgICAgICAiaGFyZF9mYWlsdXJlcyI6IFtdLAogICAgICAgICJ3YXJuaW5ncyI6IFtdLAogICAgICAgICJpbnB1dCI6IHsid2lkdGgiOiBvcmlnaW5hbF93LCAiaGVpZ2h0Ijogb3JpZ2luYWxfaH0sCiAgICAgICAgIm91dHB1dCI6IHsid2lkdGgiOiAwLCAiaGVpZ2h0IjogMH0sCiAgICAgICAgIm1lZ2FwaXhlbHMiOiAwLjAsCiAgICAgICAgImNoZWNrcyI6IHsKICAgICAgICAgICAgInJlc29sdXRpb24iOiAicGFzcyIsCiAgICAgICAgICAgICJmb3JtYXQiOiAicGFzcyIsCiAgICAgICAgICAgICJpbnRlZ3JpdHkiOiAicGFzcyIsCiAgICAgICAgICAgICJhc3BlY3RfcmF0aW8iOiAicGFzcyIsCiAgICAgICAgICAgICJ0cmFuc3BhcmVuY3kiOiAicGFzcyIsCiAgICAgICAgICAgICJzaXplIjogInBhc3MiLAogICAgICAgIH0KICAgIH0KCiAgICB0cnk6CiAgICAgICAgIyDilIDilIAgMS4gRmlsZSBJbnRlZ3JpdHkg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAgICAgdHJ5OgogICAgICAgICAgICB3aXRoIEltYWdlLm9wZW4ob3V0cHV0X3BhdGgpIGFzIGltZzoKICAgICAgICAgICAgICAgIGltZy52ZXJpZnkoKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgcmVzdWx0WyJjaGVja3MiXVsiaW50ZWdyaXR5Il0gPSAiZmFpbCIKICAgICAgICAgICAgcmVzdWx0WyJoYXJkX2ZhaWx1cmVzIl0uYXBwZW5kKGYiRmlsZSBpbnRlZ3JpdHkgY2hlY2sgZmFpbGVkOiB7ZX0iKQogICAgICAgICAgICByZXN1bHRbInBhc3NlZCJdID0gRmFsc2UKICAgICAgICAgICAgcmV0dXJuIHJlc3VsdCAgIyBDYW4ndCBwcm9jZWVkIHdpdGhvdXQgdmFsaWQgZmlsZQoKICAgICAgICAjIOKUgOKUgCAyLiBPcGVuICYgbWVhc3VyZSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgICAgICB3aXRoIEltYWdlLm9wZW4ob3V0cHV0X3BhdGgpIGFzIGltZzoKICAgICAgICAgICAgb3V0X3csIG91dF9oID0gaW1nLnNpemUKICAgICAgICAgICAgaW1nX2Zvcm1hdCA9IGltZy5mb3JtYXQgb3IgIiIKICAgICAgICAgICAgaGFzX2FscGhhID0gaW1nLm1vZGUgaW4gKCJSR0JBIiwgIkxBIiwgIlBBIikKICAgICAgICAgICAgbW9kZSA9IGltZy5tb2RlCgogICAgICAgIHJlc3VsdFsib3V0cHV0Il1bIndpZHRoIl0gPSBvdXRfdwogICAgICAgIHJlc3VsdFsib3V0cHV0Il1bImhlaWdodCJdID0gb3V0X2gKICAgICAgICBtZWdhcGl4ZWxzID0gcm91bmQoKG91dF93ICogb3V0X2gpIC8gMV8wMDBfMDAwLjAsIDIpCiAgICAgICAgcmVzdWx0WyJtZWdhcGl4ZWxzIl0gPSBtZWdhcGl4ZWxzCgogICAgICAgICMg4pSA4pSAIDMuIFJlc29sdXRpb24g4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAgICAgaWYgbWVnYXBpeGVscyA8IE1JTl9NRUdBUElYRUxTOgogICAgICAgICAgICByZXN1bHRbImNoZWNrcyJdWyJyZXNvbHV0aW9uIl0gPSAiZmFpbCIKICAgICAgICAgICAgcmVzdWx0WyJoYXJkX2ZhaWx1cmVzIl0uYXBwZW5kKAogICAgICAgICAgICAgICAgZiJSZXNvbHV0aW9uIHRvbyBsb3c6IHttZWdhcGl4ZWxzOi4yZn0gTVAgKG1pbmltdW0ge01JTl9NRUdBUElYRUxTfSBNUCByZXF1aXJlZCkiCiAgICAgICAgICAgICkKICAgICAgICAgICAgcmVzdWx0WyJwYXNzZWQiXSA9IEZhbHNlCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcmVzdWx0WyJjaGVja3MiXVsicmVzb2x1dGlvbiJdID0gInBhc3MiCgogICAgICAgICMg4pSA4pSAIDQuIEZvcm1hdCDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgICAgICBleHBlY3RlZF9mb3JtYXRzID0geyJqcGciOiB7IkpQRUcifSwgImpwZWciOiB7IkpQRUcifSwgInBuZyI6IHsiUE5HIn0sICJ3ZWJwIjogeyJXRUJQIn19CiAgICAgICAgcmVxX2ZtdF9sb3dlciA9IHJlcV9mb3JtYXQubG93ZXIoKQogICAgICAgIGFsbG93ZWQgPSBleHBlY3RlZF9mb3JtYXRzLmdldChyZXFfZm10X2xvd2VyLCB7aW1nX2Zvcm1hdH0pCiAgICAgICAgaWYgaW1nX2Zvcm1hdCBub3QgaW4gYWxsb3dlZDoKICAgICAgICAgICAgcmVzdWx0WyJjaGVja3MiXVsiZm9ybWF0Il0gPSAiZmFpbCIKICAgICAgICAgICAgcmVzdWx0WyJoYXJkX2ZhaWx1cmVzIl0uYXBwZW5kKAogICAgICAgICAgICAgICAgZiJGb3JtYXQgbWlzbWF0Y2g6IGV4cGVjdGVkIHtyZXFfZm9ybWF0LnVwcGVyKCl9LCBnb3Qge2ltZ19mb3JtYXR9IgogICAgICAgICAgICApCiAgICAgICAgICAgIHJlc3VsdFsicGFzc2VkIl0gPSBGYWxzZQoKICAgICAgICAjIOKUgOKUgCA1LiBBc3BlY3QgUmF0aW8gRHJpZnQg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAgICAgaWYgb3JpZ2luYWxfdyA+IDAgYW5kIG9yaWdpbmFsX2ggPiAwOgogICAgICAgICAgICBvcmlnX3JhdGlvID0gb3JpZ2luYWxfdyAvIG9yaWdpbmFsX2gKICAgICAgICAgICAgb3V0X3JhdGlvID0gb3V0X3cgLyBvdXRfaAogICAgICAgICAgICBkcmlmdCA9IGFicyhvcmlnX3JhdGlvIC0gb3V0X3JhdGlvKSAvIG9yaWdfcmF0aW8KICAgICAgICAgICAgaWYgZHJpZnQgPiBNQVhfQVNQRUNUX1JBVElPX0RSSUZUOgogICAgICAgICAgICAgICAgcmVzdWx0WyJjaGVja3MiXVsiYXNwZWN0X3JhdGlvIl0gPSAid2FybiIKICAgICAgICAgICAgICAgIHJlc3VsdFsid2FybmluZ3MiXS5hcHBlbmQoCiAgICAgICAgICAgICAgICAgICAgZiJBc3BlY3QgcmF0aW8gZHJpZnQgZGV0ZWN0ZWQ6IHtkcmlmdCoxMDA6LjJmfSUgKD57TUFYX0FTUEVDVF9SQVRJT19EUklGVCoxMDB9JSB0aHJlc2hvbGQpIgogICAgICAgICAgICAgICAgKQoKICAgICAgICAjIOKUgOKUgCA2LiBUcmFuc3BhcmVuY3kg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAgICAgaWYgaGFzX2FscGhhIGFuZCByZXFfZm10X2xvd2VyIGluICgianBnIiwgImpwZWciKToKICAgICAgICAgICAgcmVzdWx0WyJjaGVja3MiXVsidHJhbnNwYXJlbmN5Il0gPSAid2FybiIKICAgICAgICAgICAgcmVzdWx0WyJ3YXJuaW5ncyJdLmFwcGVuZCgiU291cmNlIGhhcyBhbHBoYSBjaGFubmVsIGJ1dCBvdXRwdXQgZm9ybWF0IGlzIEpQRUcgKHRyYW5zcGFyZW5jeSBsb3N0KSIpCgogICAgICAgICMg4pSA4pSAIDcuIEZpbGUgU2l6ZSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgICAgICBmaWxlX3NpemVfbWIgPSBvcy5wYXRoLmdldHNpemUob3V0cHV0X3BhdGgpIC8gKDEwMjQgKiAxMDI0KQogICAgICAgIGlmIGZpbGVfc2l6ZV9tYiA+IE1BWF9GSUxFX1NJWkVfTUI6CiAgICAgICAgICAgIHJlc3VsdFsiY2hlY2tzIl1bInNpemUiXSA9ICJ3YXJuIgogICAgICAgICAgICByZXN1bHRbIndhcm5pbmdzIl0uYXBwZW5kKAogICAgICAgICAgICAgICAgZiJGaWxlIHNpemUge2ZpbGVfc2l6ZV9tYjouMWZ9IE1CIGV4Y2VlZHMgQWRvYmUgU3RvY2sgbGltaXQgb2Yge01BWF9GSUxFX1NJWkVfTUJ9IE1CIgogICAgICAgICAgICApCgogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgIHJlc3VsdFsiY2hlY2tzIl1bImludGVncml0eSJdID0gImZhaWwiCiAgICAgICAgcmVzdWx0WyJoYXJkX2ZhaWx1cmVzIl0uYXBwZW5kKGYiUUMgZXhjZXB0aW9uOiB7ZX0iKQogICAgICAgIHJlc3VsdFsicGFzc2VkIl0gPSBGYWxzZQoKICAgIHJldHVybiByZXN1bHQK'
os.makedirs('/content/studio/scripts', exist_ok=True)
open('/content/studio/scripts/qc.py', 'w', encoding='utf-8').write(base64.b64decode(_b64).decode())
print('  [OK] scripts/qc.py')

# scripts/upscaler.py
_b64 = 'IiIiCnNjcmlwdHMvdXBzY2FsZXIucHkg4oCUIEFkb2JlIFN0b2NrIEFJIFN0dWRpbwoKUmVhbC1FU1JHQU4gdXBzY2FsaW5nIGVuZ2luZS4KUHJlc2VydmVkIGZyb20gb3JpZ2luYWwgd2l0aCBzdWJwcm9jZXNzIGlzb2xhdGlvbiBmb3IgVDQgVlJBTSBwcm90ZWN0aW9uLgoiIiIKCmltcG9ydCBvcwppbXBvcnQgc3lzCmltcG9ydCB0aW1lCmltcG9ydCBsb2dnaW5nCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzaHV0aWwKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gUElMIGltcG9ydCBJbWFnZQoKIyBFbnN1cmUgdG9yY2h2aXNpb24gZnVuY3Rpb25hbF90ZW5zb3IgYmFja3dhcmQgY29tcGF0aWJpbGl0eSBmb3IgYmFzaWNzcgp0cnk6CiAgICBpbXBvcnQgdG9yY2h2aXNpb24udHJhbnNmb3Jtcy5mdW5jdGlvbmFsIGFzIEYKICAgIHN5cy5tb2R1bGVzWyd0b3JjaHZpc2lvbi50cmFuc2Zvcm1zLmZ1bmN0aW9uYWxfdGVuc29yJ10gPSBGCmV4Y2VwdCBFeGNlcHRpb246CiAgICBwYXNzCgpmcm9tIHNjcmlwdHMuY29uZmlnIGltcG9ydCBURU1QX09VVFBVVF9ESVIsIFRJTEVfU0laRSwgVElMRV9QQUQsIFBSRV9QQUQKCmxvZ2dlciA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKCJBZG9iZVN0b2NrU3R1ZGlvLlVwc2NhbGVyIikKCiMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiMgRW5naW5lIGNhY2hlICYgc3VicHJvY2VzcyByZWZlcmVuY2UKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKX2VuZ2luZV9jYWNoZSA9IHt9CmFjdGl2ZV9zdWJwcm9jZXNzID0gTm9uZQoKCmRlZiBnZXRfYWN0aXZlX3N1YnByb2Nlc3MoKToKICAgIGdsb2JhbCBhY3RpdmVfc3VicHJvY2VzcwogICAgcmV0dXJuIGFjdGl2ZV9zdWJwcm9jZXNzCgoKZGVmIHNldF9hY3RpdmVfc3VicHJvY2Vzcyhwcm9jKToKICAgIGdsb2JhbCBhY3RpdmVfc3VicHJvY2VzcwogICAgYWN0aXZlX3N1YnByb2Nlc3MgPSBwcm9jCgoKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKIyBJbi1tZW1vcnkgZW5naW5lCiMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACmNsYXNzIFJlYWxFU1JHQU5FbmdpbmU6CiAgICBkZWYgX19pbml0X18oc2VsZiwgbW9kZWxfbmFtZTogc3RyID0gIlJlYWxFU1JHQU5feDRwbHVzIik6CiAgICAgICAgc2VsZi5tb2RlbF9uYW1lID0gbW9kZWxfbmFtZQogICAgICAgIHNlbGYudXBzY2FsZXIgPSBOb25lCiAgICAgICAgc2VsZi5kZXZpY2UgPSBOb25lCiAgICAgICAgc2VsZi5pc19sb2FkZWQgPSBGYWxzZQogICAgICAgIHNlbGYuaW5pdF9lcnJvciA9ICIiCgogICAgZGVmIGxvYWRfbW9kZWwoc2VsZikgLT4gYm9vbDoKICAgICAgICBpZiBzZWxmLmlzX2xvYWRlZCBhbmQgc2VsZi51cHNjYWxlciBpcyBub3QgTm9uZToKICAgICAgICAgICAgcmV0dXJuIFRydWUKCiAgICAgICAgdHJ5OgogICAgICAgICAgICBpbXBvcnQgdG9yY2gKICAgICAgICAgICAgZnJvbSByZWFsZXNyZ2FuIGltcG9ydCBSZWFsRVNSR0FOZXIKICAgICAgICAgICAgZnJvbSBiYXNpY3NyLmFyY2hzLnJyZGJuZXRfYXJjaCBpbXBvcnQgUlJEQk5ldAoKICAgICAgICAgICAgc2VsZi5kZXZpY2UgPSB0b3JjaC5kZXZpY2UoJ2N1ZGEnIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAnY3B1JykKICAgICAgICAgICAgbG9nZ2VyLmluZm8oZiJJbml0aWFsaXppbmcgUmVhbC1FU1JHQU4gaW4tbWVtb3J5IG1vZGVsIG9uIGRldmljZToge3NlbGYuZGV2aWNlfSIpCgogICAgICAgICAgICAjIEFyY2hpdGVjdHVyZSBiYXNlZCBvbiBtb2RlbCB2YXJpYW50CiAgICAgICAgICAgIGlmIHNlbGYubW9kZWxfbmFtZSA9PSAnUmVhbEVTUkdBTl94NHBsdXNfYW5pbWVfNkInOgogICAgICAgICAgICAgICAgbW9kZWwgPSBSUkRCTmV0KG51bV9pbl9jaD0zLCBudW1fb3V0X2NoPTMsIG51bV9mZWF0PTY0LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG51bV9ibG9jaz02LCBudW1fZ3Jvd19jaD0zMiwgc2NhbGU9NCkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIG1vZGVsID0gUlJEQk5ldChudW1faW5fY2g9MywgbnVtX291dF9jaD0zLCBudW1fZmVhdD02NCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBudW1fYmxvY2s9MjMsIG51bV9ncm93X2NoPTMyLCBzY2FsZT00KQoKICAgICAgICAgICAgIyBMb2NhdGUgd2VpZ2h0IGZpbGUKICAgICAgICAgICAgZmlsZW5hbWUgPSBmIntzZWxmLm1vZGVsX25hbWV9LnB0aCIKICAgICAgICAgICAgY2FuZGlkYXRlcyA9IFsKICAgICAgICAgICAgICAgIG9zLnBhdGguam9pbigiZXhwZXJpbWVudHMvcHJldHJhaW5lZF9tb2RlbHMiLCBmaWxlbmFtZSksCiAgICAgICAgICAgICAgICBvcy5wYXRoLmpvaW4oIndlaWdodHMiLCBmaWxlbmFtZSksCiAgICAgICAgICAgICAgICBvcy5wYXRoLmpvaW4oIlJlYWwtRVNSR0FOL2V4cGVyaW1lbnRzL3ByZXRyYWluZWRfbW9kZWxzIiwgZmlsZW5hbWUpLAogICAgICAgICAgICAgICAgb3MucGF0aC5qb2luKCIvY29udGVudC9VcHNjYWxlLUFJL2V4cGVyaW1lbnRzL3ByZXRyYWluZWRfbW9kZWxzIiwgZmlsZW5hbWUpLAogICAgICAgICAgICAgICAgb3MucGF0aC5qb2luKCIvY29udGVudC9SZWFsLUVTUkdBTi9leHBlcmltZW50cy9wcmV0cmFpbmVkX21vZGVscyIsIGZpbGVuYW1lKSwKICAgICAgICAgICAgXQogICAgICAgICAgICBtb2RlbF9wYXRoID0gTm9uZQogICAgICAgICAgICBmb3IgYyBpbiBjYW5kaWRhdGVzOgogICAgICAgICAgICAgICAgaWYgb3MucGF0aC5leGlzdHMoYyk6CiAgICAgICAgICAgICAgICAgICAgbW9kZWxfcGF0aCA9IG9zLnBhdGguYWJzcGF0aChjKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCgogICAgICAgICAgICBpZiBub3QgbW9kZWxfcGF0aDoKICAgICAgICAgICAgICAgIHNlbGYuaW5pdF9lcnJvciA9IGYiV2VpZ2h0IGZpbGUgJ3tmaWxlbmFtZX0nIG5vdCBmb3VuZCBvbiBkaXNrLiIKICAgICAgICAgICAgICAgIGxvZ2dlci5lcnJvcihzZWxmLmluaXRfZXJyb3IpCiAgICAgICAgICAgICAgICByZXR1cm4gRmFsc2UKCiAgICAgICAgICAgIHVzZV9oYWxmID0gdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKQogICAgICAgICAgICBzZWxmLnVwc2NhbGVyID0gUmVhbEVTUkdBTmVyKAogICAgICAgICAgICAgICAgc2NhbGU9NCwKICAgICAgICAgICAgICAgIG1vZGVsX3BhdGg9bW9kZWxfcGF0aCwKICAgICAgICAgICAgICAgIGRuaV93ZWlnaHQ9Tm9uZSwKICAgICAgICAgICAgICAgIG1vZGVsPW1vZGVsLAogICAgICAgICAgICAgICAgdGlsZT1USUxFX1NJWkUsCiAgICAgICAgICAgICAgICB0aWxlX3BhZD1USUxFX1BBRCwKICAgICAgICAgICAgICAgIHByZV9wYWQ9UFJFX1BBRCwKICAgICAgICAgICAgICAgIGhhbGY9dXNlX2hhbGYsCiAgICAgICAgICAgICAgICBncHVfaWQ9MCBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgTm9uZSwKICAgICAgICAgICAgKQogICAgICAgICAgICBzZWxmLmlzX2xvYWRlZCA9IFRydWUKICAgICAgICAgICAgbG9nZ2VyLmluZm8oIlJlYWwtRVNSR0FOIGVuZ2luZSBsb2FkZWQgc3VjY2Vzc2Z1bGx5LiIpCiAgICAgICAgICAgIHJldHVybiBUcnVlCgogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgc2VsZi5pbml0X2Vycm9yID0gc3RyKGUpCiAgICAgICAgICAgIGxvZ2dlci5lcnJvcihmIkZhaWxlZCB0byBsb2FkIFJlYWwtRVNSR0FOIGVuZ2luZToge2V9IikKICAgICAgICAgICAgcmV0dXJuIEZhbHNlCgogICAgZGVmIHVwc2NhbGUoc2VsZiwgaW5wdXRfcGF0aDogc3RyLCBvdXRwdXRfcGF0aDogc3RyLCBzY2FsZTogZmxvYXQgPSA0KSAtPiBib29sOgogICAgICAgIGlmIG5vdCBzZWxmLmxvYWRfbW9kZWwoKToKICAgICAgICAgICAgcmV0dXJuIEZhbHNlCiAgICAgICAgdHJ5OgogICAgICAgICAgICBpbXBvcnQgY3YyCiAgICAgICAgICAgIGltcG9ydCBudW1weSBhcyBucAoKICAgICAgICAgICAgaW1nID0gY3YyLmltcmVhZChpbnB1dF9wYXRoLCBjdjIuSU1SRUFEX1VOQ0hBTkdFRCkKICAgICAgICAgICAgaWYgaW1nIGlzIE5vbmU6CiAgICAgICAgICAgICAgICBsb2dnZXIuZXJyb3IoZiJDb3VsZCBub3QgcmVhZCBpbWFnZToge2lucHV0X3BhdGh9IikKICAgICAgICAgICAgICAgIHJldHVybiBGYWxzZQoKICAgICAgICAgICAgb3V0cHV0LCBfID0gc2VsZi51cHNjYWxlci5lbmhhbmNlKGltZywgb3V0c2NhbGU9c2NhbGUpCiAgICAgICAgICAgIGN2Mi5pbXdyaXRlKG91dHB1dF9wYXRoLCBvdXRwdXQpCiAgICAgICAgICAgIGxvZ2dlci5pbmZvKGYiSW4tbWVtb3J5IHVwc2NhbGUgY29tcGxldGUg4oaSIHtvdXRwdXRfcGF0aH0iKQogICAgICAgICAgICByZXR1cm4gVHJ1ZQoKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgIGxvZ2dlci5lcnJvcihmIkluLW1lbW9yeSB1cHNjYWxlIGVycm9yOiB7ZX0iKQogICAgICAgICAgICByZXR1cm4gRmFsc2UKCiAgICBkZWYgdW5sb2FkKHNlbGYpOgogICAgICAgICIiIlJlbGVhc2UgR1BVIG1lbW9yeSBhZnRlciB1cHNjYWxpbmcuIiIiCiAgICAgICAgdHJ5OgogICAgICAgICAgICBpZiBzZWxmLnVwc2NhbGVyIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgZGVsIHNlbGYudXBzY2FsZXIKICAgICAgICAgICAgICAgIHNlbGYudXBzY2FsZXIgPSBOb25lCiAgICAgICAgICAgICAgICBzZWxmLmlzX2xvYWRlZCA9IEZhbHNlCiAgICAgICAgICAgIGltcG9ydCB0b3JjaAogICAgICAgICAgICBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgICAgICAgICAgdG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpCiAgICAgICAgICAgIGxvZ2dlci5pbmZvKCJSZWFsLUVTUkdBTiBlbmdpbmUgdW5sb2FkZWQsIFZSQU0gY2xlYXJlZC4iKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgbG9nZ2VyLndhcm5pbmcoZiJVbmxvYWQgd2FybmluZzoge2V9IikKCgpkZWYgZ2V0X2VuZ2luZShtb2RlbF9uYW1lOiBzdHIgPSAiUmVhbEVTUkdBTl94NHBsdXMiKSAtPiBSZWFsRVNSR0FORW5naW5lOgogICAgaWYgbW9kZWxfbmFtZSBub3QgaW4gX2VuZ2luZV9jYWNoZToKICAgICAgICBfZW5naW5lX2NhY2hlW21vZGVsX25hbWVdID0gUmVhbEVTUkdBTkVuZ2luZShtb2RlbF9uYW1lKQogICAgcmV0dXJuIF9lbmdpbmVfY2FjaGVbbW9kZWxfbmFtZV0KCgojIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAojIFN1YnByb2Nlc3MgQ0xJIGZhbGxiYWNrIChpc29sYXRpb24gbW9kZSkKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKZGVmIF9maW5kX3JlYWxlc3JnYW5fc2NyaXB0KCkgLT4gc3RyIHwgTm9uZToKICAgIGNhbmRpZGF0ZXMgPSBbCiAgICAgICAgImluZmVyZW5jZV9yZWFsZXNyZ2FuLnB5IiwKICAgICAgICAiL2NvbnRlbnQvVXBzY2FsZS1BSS9pbmZlcmVuY2VfcmVhbGVzcmdhbi5weSIsCiAgICAgICAgIi9jb250ZW50L1JlYWwtRVNSR0FOL2luZmVyZW5jZV9yZWFsZXNyZ2FuLnB5IiwKICAgIF0KICAgIGZvciBjIGluIGNhbmRpZGF0ZXM6CiAgICAgICAgaWYgb3MucGF0aC5leGlzdHMoYyk6CiAgICAgICAgICAgIHJldHVybiBjCiAgICByZXR1cm4gTm9uZQoKCmRlZiBydW5fdXBzY2FsZV9zdWJwcm9jZXNzKAogICAgaW5wdXRfcGF0aDogc3RyLAogICAgb3V0cHV0X2Rpcjogc3RyLAogICAgc2NhbGU6IGZsb2F0LAogICAgbW9kZWxfbmFtZTogc3RyLAogICAgZXh0OiBzdHIsCikgLT4gdHVwbGVbYm9vbCwgc3RyXToKICAgICIiIgogICAgUnVuIFJlYWwtRVNSR0FOIGFzIGEgc3VicHJvY2Vzcy4KICAgIFRoaXMgcHJvdmlkZXMgc3VicHJvY2VzcyBpc29sYXRpb24g4oCUIGlmIFZSQU0gY3Jhc2hlcywgRmFzdEFQSSBzdGF5cyBhbGl2ZS4KICAgIFJldHVybnMgKHN1Y2Nlc3MsIG91dHB1dF9maWxlX3BhdGgpCiAgICAiIiIKICAgIGdsb2JhbCBhY3RpdmVfc3VicHJvY2VzcwoKICAgIHNjcmlwdCA9IF9maW5kX3JlYWxlc3JnYW5fc2NyaXB0KCkKICAgIGlmIG5vdCBzY3JpcHQ6CiAgICAgICAgcmV0dXJuIEZhbHNlLCAiIgoKICAgIG9zLm1ha2VkaXJzKG91dHB1dF9kaXIsIGV4aXN0X29rPVRydWUpCgogICAgY21kID0gWwogICAgICAgICJweXRob24iLCBzY3JpcHQsCiAgICAgICAgIi1uIiwgbW9kZWxfbmFtZSwKICAgICAgICAiLWkiLCBpbnB1dF9wYXRoLAogICAgICAgICItbyIsIG91dHB1dF9kaXIsCiAgICAgICAgIi1zIiwgc3RyKHNjYWxlKSwKICAgICAgICAiLS1leHQiLCBleHQsCiAgICAgICAgIi0tdGlsZSIsIHN0cihUSUxFX1NJWkUpLAogICAgICAgICItLXRpbGVfcGFkIiwgc3RyKFRJTEVfUEFEKSwKICAgICAgICAiLS1wcmVfcGFkIiwgc3RyKFBSRV9QQUQpLAogICAgXQoKICAgIHRyeToKICAgICAgICBpbXBvcnQgdG9yY2gKICAgICAgICBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgICAgICBjbWQuYXBwZW5kKCItLWhhbGYiKQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICBwYXNzCgogICAgbG9nZ2VyLmluZm8oZiJSdW5uaW5nIFJlYWwtRVNSR0FOIHN1YnByb2Nlc3M6IHsnICcuam9pbihjbWQpfSIpCgogICAgdHJ5OgogICAgICAgIHByb2MgPSBzdWJwcm9jZXNzLlBvcGVuKGNtZCwgc3Rkb3V0PXN1YnByb2Nlc3MuUElQRSwgc3RkZXJyPXN1YnByb2Nlc3MuUElQRSwgdGV4dD1UcnVlKQogICAgICAgIGFjdGl2ZV9zdWJwcm9jZXNzID0gcHJvYwogICAgICAgIHN0ZG91dCwgc3RkZXJyID0gcHJvYy5jb21tdW5pY2F0ZSgpCiAgICAgICAgYWN0aXZlX3N1YnByb2Nlc3MgPSBOb25lCgogICAgICAgIGlmIHByb2MucmV0dXJuY29kZSAhPSAwOgogICAgICAgICAgICBsb2dnZXIuZXJyb3IoZiJSZWFsLUVTUkdBTiBzdWJwcm9jZXNzIGZhaWxlZCAoZXhpdCB7cHJvYy5yZXR1cm5jb2RlfSk6IHtzdGRlcnJ9IikKICAgICAgICAgICAgcmV0dXJuIEZhbHNlLCAiIgoKICAgICAgICAjIEZpbmQgb3V0cHV0IGZpbGUKICAgICAgICBiYXNlID0gb3MucGF0aC5zcGxpdGV4dChvcy5wYXRoLmJhc2VuYW1lKGlucHV0X3BhdGgpKVswXQogICAgICAgIGV4cGVjdGVkID0gb3MucGF0aC5qb2luKG91dHB1dF9kaXIsIGYie2Jhc2V9X291dC57ZXh0fSIpCiAgICAgICAgaWYgb3MucGF0aC5leGlzdHMoZXhwZWN0ZWQpOgogICAgICAgICAgICByZXR1cm4gVHJ1ZSwgZXhwZWN0ZWQKCiAgICAgICAgIyBUcnkgYWx0ZXJuYXRlIG5hbWluZwogICAgICAgIGZvciBmbmFtZSBpbiBvcy5saXN0ZGlyKG91dHB1dF9kaXIpOgogICAgICAgICAgICBpZiBiYXNlIGluIGZuYW1lIGFuZCBmbmFtZS5lbmRzd2l0aChmIi57ZXh0fSIpOgogICAgICAgICAgICAgICAgcmV0dXJuIFRydWUsIG9zLnBhdGguam9pbihvdXRwdXRfZGlyLCBmbmFtZSkKCiAgICAgICAgbG9nZ2VyLmVycm9yKCJSZWFsLUVTUkdBTiBzdWJwcm9jZXNzIHJhbiBidXQgb3V0cHV0IGZpbGUgbm90IGZvdW5kLiIpCiAgICAgICAgcmV0dXJuIEZhbHNlLCAiIgoKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICBhY3RpdmVfc3VicHJvY2VzcyA9IE5vbmUKICAgICAgICBsb2dnZXIuZXJyb3IoZiJTdWJwcm9jZXNzIGV4Y2VwdGlvbjoge2V9IikKICAgICAgICByZXR1cm4gRmFsc2UsICIiCgoKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKIyBNb2NrIGZhbGxiYWNrIChmb3IgdGVzdGluZyB3aXRob3V0IEdQVSkKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKZGVmIHJ1bl91cHNjYWxlX21vY2soaW5wdXRfcGF0aDogc3RyLCBvdXRwdXRfcGF0aDogc3RyLCBzY2FsZTogZmxvYXQpIC0+IGJvb2w6CiAgICAiIiJQSUwtYmFzZWQgbW9jayB1cHNjYWxlciBmb3IgZW52aXJvbm1lbnRzIHdpdGhvdXQgR1BVL1JlYWwtRVNSR0FOLiIiIgogICAgdHJ5OgogICAgICAgIGxvZ2dlci5pbmZvKGYiW01PQ0tdIFVwc2NhbGluZyB7aW5wdXRfcGF0aH0gKHNjYWxlPXtzY2FsZX0pIikKICAgICAgICB0aW1lLnNsZWVwKDEuNSkgICMgU2ltdWxhdGUgcHJvY2Vzc2luZwogICAgICAgIGltZyA9IEltYWdlLm9wZW4oaW5wdXRfcGF0aCkKICAgICAgICB3LCBoID0gaW1nLnNpemUKICAgICAgICBuZXdfdywgbmV3X2ggPSBpbnQodyAqIHNjYWxlKSwgaW50KGggKiBzY2FsZSkKICAgICAgICBvdXQgPSBpbWcucmVzaXplKChuZXdfdywgbmV3X2gpLCBJbWFnZS5SZXNhbXBsaW5nLkxBTkNaT1MpCiAgICAgICAgZXh0ID0gUGF0aChvdXRwdXRfcGF0aCkuc3VmZml4Lmxvd2VyKCkKICAgICAgICBpZiBleHQgaW4gKCIuanBnIiwgIi5qcGVnIik6CiAgICAgICAgICAgIG91dC5jb252ZXJ0KCJSR0IiKS5zYXZlKG91dHB1dF9wYXRoLCAiSlBFRyIsIHF1YWxpdHk9OTUpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgb3V0LnNhdmUob3V0cHV0X3BhdGgsICJQTkciKQogICAgICAgIHJldHVybiBUcnVlCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgbG9nZ2VyLmVycm9yKGYiTW9jayB1cHNjYWxlIGZhaWxlZDoge2V9IikKICAgICAgICByZXR1cm4gRmFsc2UKCgojIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAojIFByaW1hcnkgZW50cnkgcG9pbnQKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKZGVmIHJ1bl91cHNjYWxlKAogICAgaW5wdXRfcGF0aDogc3RyLAogICAgb3V0cHV0X3BhdGg6IHN0ciwKICAgIHNjYWxlOiBmbG9hdCwKICAgIG1vZGVsX25hbWU6IHN0ciwKICAgIGV4dDogc3RyLAogICAgcXVhbGl0eTogaW50ID0gOTUsCiAgICBwcm9ncmVzc19jYWxsYmFjaz1Ob25lLAopIC0+IGJvb2w6CiAgICAiIiIKICAgIFVwc2NhbGUgYW4gaW1hZ2UgdXNpbmcgdGhlIGJlc3QgYXZhaWxhYmxlIG1ldGhvZDoKICAgIDEuIEluLW1lbW9yeSBSZWFsRVNSR0FOZXIgKGZhc3Rlc3QsIGRpcmVjdCBQeXRob24gQVBJKQogICAgMi4gU3VicHJvY2VzcyBDTEkgKHN1YnByb2Nlc3MgaXNvbGF0aW9uIGZhbGxiYWNrKQogICAgMy4gUElMIG1vY2sgKHRlc3Rpbmcgb25seSkKCiAgICBwcm9ncmVzc19jYWxsYmFjazogb3B0aW9uYWwgY2FsbGFibGUocGVyY2VudDogaW50KQogICAgIiIiCiAgICBpZiBwcm9ncmVzc19jYWxsYmFjazoKICAgICAgICBwcm9ncmVzc19jYWxsYmFjayg1KQoKICAgICMgTWV0aG9kIDE6IEluLW1lbW9yeQogICAgdHJ5OgogICAgICAgIGZyb20gcmVhbGVzcmdhbiBpbXBvcnQgUmVhbEVTUkdBTmVyICAjIG5vcWE6IEY0MDEKICAgICAgICBlbmdpbmUgPSBnZXRfZW5naW5lKG1vZGVsX25hbWUpCiAgICAgICAgaWYgcHJvZ3Jlc3NfY2FsbGJhY2s6CiAgICAgICAgICAgIHByb2dyZXNzX2NhbGxiYWNrKDIwKQogICAgICAgIHN1Y2Nlc3MgPSBlbmdpbmUudXBzY2FsZShpbnB1dF9wYXRoLCBvdXRwdXRfcGF0aCwgc2NhbGUpCiAgICAgICAgaWYgc3VjY2VzczoKICAgICAgICAgICAgaWYgcHJvZ3Jlc3NfY2FsbGJhY2s6CiAgICAgICAgICAgICAgICBwcm9ncmVzc19jYWxsYmFjaygxMDApCiAgICAgICAgICAgIHJldHVybiBUcnVlCiAgICAgICAgbG9nZ2VyLndhcm5pbmcoIkluLW1lbW9yeSBlbmdpbmUgZmFpbGVkLCB0cnlpbmcgc3VicHJvY2Vzcy4uLiIpCiAgICBleGNlcHQgSW1wb3J0RXJyb3I6CiAgICAgICAgbG9nZ2VyLmluZm8oInJlYWxlc3JnYW4gbm90IGltcG9ydGFibGUsIHRyeWluZyBzdWJwcm9jZXNzLi4uIikKCiAgICBpZiBwcm9ncmVzc19jYWxsYmFjazoKICAgICAgICBwcm9ncmVzc19jYWxsYmFjaygzMCkKCiAgICAjIE1ldGhvZCAyOiBTdWJwcm9jZXNzCiAgICBvdXRfZGlyID0gc3RyKFBhdGgob3V0cHV0X3BhdGgpLnBhcmVudCkKICAgIHN1Y2Nlc3MsIGZvdW5kX3BhdGggPSBydW5fdXBzY2FsZV9zdWJwcm9jZXNzKGlucHV0X3BhdGgsIG91dF9kaXIsIHNjYWxlLCBtb2RlbF9uYW1lLCBleHQpCiAgICBpZiBzdWNjZXNzIGFuZCBmb3VuZF9wYXRoOgogICAgICAgIGlmIGZvdW5kX3BhdGggIT0gb3V0cHV0X3BhdGg6CiAgICAgICAgICAgIHNodXRpbC5tb3ZlKGZvdW5kX3BhdGgsIG91dHB1dF9wYXRoKQogICAgICAgIGlmIHByb2dyZXNzX2NhbGxiYWNrOgogICAgICAgICAgICBwcm9ncmVzc19jYWxsYmFjaygxMDApCiAgICAgICAgcmV0dXJuIFRydWUKCiAgICBsb2dnZXIud2FybmluZygiU3VicHJvY2VzcyBmYWlsZWQuIFVzaW5nIG1vY2sgUElMIGZhbGxiYWNrLiIpCiAgICBpZiBwcm9ncmVzc19jYWxsYmFjazoKICAgICAgICBwcm9ncmVzc19jYWxsYmFjayg1MCkKCiAgICAjIE1ldGhvZCAzOiBNb2NrCiAgICByZXN1bHQgPSBydW5fdXBzY2FsZV9tb2NrKGlucHV0X3BhdGgsIG91dHB1dF9wYXRoLCBzY2FsZSkKICAgIGlmIHJlc3VsdCBhbmQgcHJvZ3Jlc3NfY2FsbGJhY2s6CiAgICAgICAgcHJvZ3Jlc3NfY2FsbGJhY2soMTAwKQogICAgcmV0dXJuIHJlc3VsdAo='
os.makedirs('/content/studio/scripts', exist_ok=True)
open('/content/studio/scripts/upscaler.py', 'w', encoding='utf-8').write(base64.b64decode(_b64).decode())
print('  [OK] scripts/upscaler.py')

# scripts/ollama_vision.py
_b64 = '...'
os.makedirs('/content/studio/scripts', exist_ok=True)
open('/content/studio/scripts/ollama_vision.py', 'w', encoding='utf-8').write(base64.b64decode(_b64).decode())
print('  [OK] scripts/ollama_vision.py')

# scripts/batch_processor.py
_b64 = '...'
os.makedirs('/content/studio/scripts', exist_ok=True)
open('/content/studio/scripts/batch_processor.py', 'w', encoding='utf-8').write(base64.b64decode(_b64).decode())
print('  [OK] scripts/batch_processor.py')

print('[OK] All scripts written')


In [ ]:
# =======================================================
# CELL 12: Write FastAPI Backend (app/main.py)
# =======================================================
import base64, os
STUDIO = '/content/studio'
os.makedirs(f'{STUDIO}/app', exist_ok=True)
_b64 = '...'
open(f'{STUDIO}/app/main.py', 'w', encoding='utf-8').write(
    base64.b64decode(_b64).decode('utf-8')
)
# app/__init__.py
open(f'{STUDIO}/app/__init__.py', 'w').write('')
print('[OK] app/main.py written')


In [ ]:
# =======================================================
# CELL 13: Write Frontend (app/index.html)
# =======================================================
import base64, os
STUDIO = '/content/studio'
_b64 = '...'
open(f'{STUDIO}/app/index.html', 'w', encoding='utf-8').write(
    base64.b64decode(_b64).decode('utf-8')
)
print('[OK] app/index.html written')
print(f'  Size: {len(base64.b64decode(_b64)):,} bytes')


In [ ]:
# =======================================================
# CELL 14: Write requirements.txt + Verify File Structure
# =======================================================
import base64, os, sys
STUDIO = '/content/studio'

# requirements.txt
_b64 = 'ZmFzdGFwaT49MC45NS4wCnV2aWNvcm4+PTAuMjIuMApweXRob24tbXVsdGlwYXJ0Pj0wLjAuNgpwc3V0aWw+PTUuOS4wCnBpbGxvdz49MTAuMC4wCmh0dHB4Pj0wLjI0LjAKYWlvZmlsZXM+PTIzLjAuMApweWRhbnRpYz49Mi4wLjAK'
open(f'{STUDIO}/requirements.txt', 'w').write(
    base64.b64decode(_b64).decode('utf-8')
)

# Add studio dir to Python path
if STUDIO not in sys.path:
    sys.path.insert(0, STUDIO)

# List structure
for root, dirs, files in os.walk(STUDIO):
    dirs[:] = [d for d in dirs if d not in ['__pycache__','temp_output','uploads','output','metadata','logs','archives','failed']]
    level = root.replace(STUDIO, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        print(f'{indent}  {f}')

print()
print('[OK] File structure verified')


In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 15: Start FastAPI Backend
# ═══════════════════════════════════════════════════════
import subprocess, sys, os, time, requests

STUDIO = '/content/studio'

# Ensure torchvision compat for the server process
env = os.environ.copy()
env['PYTHONPATH'] = STUDIO

print("Starting FastAPI (uvicorn)…")
server_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app.main:app',
     '--host', '0.0.0.0', '--port', '8000',
     '--workers', '1', '--log-level', 'warning'],
    cwd=STUDIO,
    env=env,
)

# Wait for backend to be ready
for i in range(45):
    try:
        r = requests.get('http://localhost:8000/api/health', timeout=2)
        if r.ok:
            h = r.json()
            print(f"✓ FastAPI running (took {i+1}s)")
            print(f"  GPU     : {h.get('gpu_name','—')}")
            print(f"  Ollama  : {'Ready (' + h.get('ollama_model','?') + ')' if h.get('ollama_ready') else 'Initializing…'}")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print("⚠ Backend may still be starting. Check output above.")
    print("  Run: !curl http://localhost:8000/api/health")


In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 16: Cloudflare Tunnel + Final URL
# ═══════════════════════════════════════════════════════
import subprocess, threading, time, re, shutil, requests

# Install cloudflared if missing
if not shutil.which('cloudflared'):
    print("Installing cloudflared…")
    subprocess.run([
        'wget', '-q', '-O', '/usr/local/bin/cloudflared',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    ], check=True)
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)
    print("✓ cloudflared installed")

public_url = None
url_event = threading.Event()

def _run_tunnel():
    global public_url
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
        if m and not public_url:
            public_url = m.group(0)
            url_event.set()

t = threading.Thread(target=_run_tunnel, daemon=True)
t.start()

print("Waiting for Cloudflare tunnel…")
url_event.wait(timeout=60)

if not public_url:
    print("⚠ Cloudflare URL not received within 60s.")
    print("  Try: !cloudflared tunnel --url http://localhost:8000")
else:
    # Health check through public tunnel
    time.sleep(3)
    try:
        hr = requests.get(f'{public_url}/api/health', timeout=20)
        health_ok = hr.ok
    except Exception:
        health_ok = False

    bar = "=" * 57
    print()
    print(bar)
    print("  ADOBE STOCK AI STUDIO — READY")
    print(bar)
    print()
    print(f"  T4 GPU       : ✓ Connected")
    print(f"  Real-ESRGAN  : ✓ Ready")
    print(f"  Ollama       : ✓ {active_model}")
    print(f"  FastAPI      : ✓ Running :8000")
    print(f"  Tunnel       : ✓ Active")
    print(f"  Health Check : {'✓ PASS' if health_ok else '⚠ Check manually'}")
    print()
    print(f"  ┌{'─'*53}┐")
    print(f"  │  🌐  OPEN THIS URL IN YOUR BROWSER:              │")
    print(f"  │  {public_url:<51}  │")
    print(f"  └{'─'*53}┘")
    print()
    print("  Instructions:")
    print("  1. Open the URL above")
    print("  2. Drop images in the upload zone (all at once)")
    print("  3. Wait for 100/100 Uploaded ✓")
    print("  4. Click  ▶ Start Processing")
    print("  5. Watch real-time progress")
    print("  6. Download ZIP / CSV / JSON when complete")
    print()
    print(bar)
